# Spotify artist recommender — project writeup

*Author: Ryan Quinlan. Status: in progress.*

A walk through the decision tree behind this project: what I built, why, what works, and what doesn't. Written for someone reading this cold (e.g. a co-op interviewer) — assumes no project context.

**Read order:**
1. The problem and why it's non-trivial in 2025
2. The data
3. The model
4. The cold-start problem and proxy substitution
5. Genre enrichment and content fallback
6. The live system
7. Limitations and lessons
8. What's next

## 1. The problem

TODO: Frame the goal in one paragraph — "build artist recommendations from a user's Spotify listening."

Key context to surface:
- Spotify deprecated `/recommendations` and `/related-artists` in Nov 2024, along with audio features. The endpoints that *would* have made this trivial don't exist anymore.
- Spotify's public API gives recent activity and top artists, but no streaming history beyond the manual data export. So the recommender has to be built from scratch on top of whatever signal the live API provides.
- This is the constraint that turns a 2-hour script into a real ML project.

## 2. The data

TODO: Last.fm 360K dataset — 358,872 users × 267,739 artists, density 0.0181%. Why this dataset (the only large-scale public implicit-feedback music dataset). The catch: it's a 2008–2009 snapshot.

**Coverage chart**: what fraction of *my own* recent listening falls outside the dataset? This is the load-bearing chart of the writeup.

In [ ]:
# TODO: chart — % of plays in user's listening history that match an artist in CF vocab,
#       broken down by year-of-release or by play count.
# Already established: ~50% play-weighted match rate on user's own data.
# Visualize as: bar chart of matched vs unmatched, with example artists in each bucket.

## 3. The model — ALS on implicit feedback

TODO:
- Why matrix factorization, why implicit-feedback variant (Hu/Koren/Volinsky 2008).
- LensKit 2026.1 `ImplicitMFScorer`. Hyperparameters: 64 factors, 10 epochs, alpha=40, regularization=0.1.
- One paragraph on the math: latent user/item embeddings, confidence-weighted least squares.
- Sanity-check the model: nearest-neighbor probes (Radiohead → Aphex Twin / BoC; Kanye → Jay-Z / Kid Cudi). Show the model learned something coherent.

In [ ]:
# TODO: nearest-neighbor table for ~5 well-known artists.
# Source the function from notebooks/02_explore_recommender.ipynb.

## 4. The cold-start problem and proxy substitution

TODO:
- Real users don't exist in the 2009 dataset. We do **fold-in inference**: build a synthetic user vector at request time from their Top Artists, fold into the trained latent space (single matrix solve, sub-100ms).
- But ~50% of their listening is for artists not in the CF vocab either (Tyler, The Creator; Frank Ocean; Kendrick; etc.). Those artists can't go directly into the user vector.
- **Proxy substitution**: query Last.fm `artist.getSimilar`, intersect with CF vocab, contribute the matched proxies to the user vector at a damped weight. Validate on weak (HYUKOH, Malcolm Todd) vs strong (Tyler, Frank, Kendrick) cases.

**The lesson worth telling**: I initially passed proxy weights via the `score` field on LensKit's `ItemList`. Decay sweeps showed identical recs at every value — the field is silently ignored unless `use_ratings=True` and weights are passed via `rating=`. Found by reading LensKit source. Good story for an interviewer about debugging by source-reading.

In [ ]:
# TODO: chart or table — proxy coverage by target artist.
# Strong: Tyler (17/50), Frank (17/50), Kendrick (22/50).
# Weak:   HYUKOH (4/50), Malcolm Todd (2/50).
# Then a side-by-side: top-10 recs for Tyler via direct-CF (impossible) vs proxy-fold-in.
# Show the proxy approach produces a coherent rec neighborhood.

## 5. Genre enrichment and content fallback

TODO:
- When proxy substitution still leaves the user vector too sparse (e.g. a listener entirely on post-2020 niche artists), we fall back to a content-based scorer that ranks by genre-tag overlap.
- Genre signal pulled from multiple sources, in priority order: HF Spotify Tracks dataset → Last.fm `getTopTags` → Spotify API genres → Wikidata.
- Why multi-source: any single source is sparse and unreliable for some artists (Spotify routinely empty for big artists like Tyler).
- Tag normalization and denylist: Last.fm tags include junk (`seen live`, year tags, mood words). Filtered against a 125-genre allowlist.

## 6. The live system

TODO:
- Streamlit app, OAuth via Spotify (auth code flow with PKCE).
- Pipeline at request time: fetch top_artists(short_term + medium_term) → fuzzy-match against CF vocab → unmatched routed through proxy substitution → fold-in solve → top-N.
- Coverage gate: if post-routing the user vector has fewer than ~5 items in CF vocab, skip the CF path entirely and serve content-based recs.
- Architecture diagram (block diagram of components).

In [ ]:
# TODO: end-to-end demo cell.
# Pick a user (yourself). Walk through:
#  - Top short_term artists fetched from Spotify
#  - Which were directly matched, which went through proxy
#  - Final user vector
#  - Top-10 recs
# This is the demo a reader actually wants to see.

## 7. Limitations and lessons

TODO — this section is what makes the writeup feel honest:
- **2009 cutoff is a structural ceiling**, not a bug. No amount of normalization will help if Tyler isn't in the dataset.
- **Popularity bias** in raw ALS recs (Smash Mouth, Katy Perry surface). Mitigated by fold-in for the live path; less of an issue with focused seed lists.
- **Junk artists** (`[unknown]`, `various artists`, `soundtrack`) dominate naive ALS scores; needed an explicit filter list.
- **Fuzzy matching is dangerous below threshold ≈95.** `token_set_ratio` matched 'Malcolm Todd' to 'todd'. `fuzz.ratio` at 92 misrouted George Clanton to George Clinton.
- **`use_ratings=False` silently dropped my fold-in weights.** Caught only by reading LensKit source after a no-op decay sweep.
- **Spotify dev-mode cap of 25 manually allowlisted users.** Production approval is a 2–6 week review. Demo mode + screencast is the portfolio workaround.

## 8. What's next

TODO:
- Two-hop proxy substitution for weak cases (HYUKOH, Malcolm Todd).
- Retraining with `use_ratings=True` to actually use play counts as confidence weights.
- Submit Spotify Extension review for production access.